# 01 — Conversion Explorer

Convert quantum algorithms to ZX diagrams at multiple simplification levels.
Build intuition for how ZX rewrites reshape algorithmic structure.

In [ ]:
import pandas as pd
import pyzx as zx
import matplotlib.pyplot as plt

from zx_motifs.algorithms.registry import REGISTRY
from zx_motifs.pipeline.converter import convert_at_all_levels
from zx_motifs.pipeline.featurizer import pyzx_to_networkx, compute_graph_features

## Batch Conversion

Convert every algorithm in the registry at multiple qubit counts.

In [ ]:
all_snapshots = []
for entry in REGISTRY:
    min_q, max_q = entry.qubit_range
    for n in range(min_q, min(max_q + 1, 6)):  # Cap at 5 qubits for exploration
        try:
            qc = entry.generator(n_qubits=n)
            snaps = convert_at_all_levels(qc, f"{entry.name}_q{n}")
            all_snapshots.extend(snaps)
            print(f"  {entry.name} (n={n}): {len(snaps)} snapshots")
        except Exception as e:
            print(f"  {entry.name} (n={n}): FAILED — {e}")

print(f"\nTotal snapshots: {len(all_snapshots)}")

## Summary Statistics

In [ ]:
rows = [s.to_dict() for s in all_snapshots]
df = pd.DataFrame(rows)
print(df.groupby("level")[["vertices", "edges", "t_gates"]].mean().round(1))

## Simplification Cascade Visualization

Pick one algorithm and visualize how each simplification level transforms it.

In [ ]:
target_algo = "grover_q3"
target_snaps = [s for s in all_snapshots if s.algorithm_name == target_algo]

if target_snaps:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    for ax, snap in zip(axes.flat, target_snaps):
        ax.set_title(f"{snap.level.value}\nV={snap.num_vertices} E={snap.num_edges}")
        zx.draw(snap.graph, ax=ax)
    plt.suptitle(f"Simplification Cascade: {target_algo}", fontsize=16)
    plt.tight_layout()
    plt.savefig("simplification_cascade.png", dpi=150)
    plt.show()
else:
    print(f"Algorithm '{target_algo}' not found in snapshots.")

## Feature Extraction & Clustering

In [ ]:
# Convert all snapshots to NetworkX and compute features
feature_rows = []
nx_graphs = {}  # (algo_name, level) -> nx.Graph

for snap in all_snapshots:
    nxg = pyzx_to_networkx(snap.graph, coarsen_phases=True)
    key = (snap.algorithm_name, snap.level.value)
    nx_graphs[key] = nxg

    feats = compute_graph_features(nxg)
    feats["algorithm"] = snap.algorithm_name
    feats["level"] = snap.level.value
    feature_rows.append(feats)

feat_df = pd.DataFrame(feature_rows)
print(feat_df[["algorithm", "level", "n_nodes", "n_edges",
               "hadamard_ratio", "avg_degree"]].head(20).to_string())

In [ ]:
try:
    import seaborn as sns

    spider_df = feat_df[feat_df["level"] == "spider_fused"].copy()
    cluster_cols = ["n_nodes", "n_edges", "hadamard_ratio", "avg_degree", "density"]
    available_cols = [c for c in cluster_cols if c in spider_df.columns]

    if len(available_cols) >= 2:
        # Extract base algorithm name for hue
        spider_df["algo_base"] = spider_df["algorithm"].str.rsplit("_q", n=1).str[0]
        sns.pairplot(spider_df, vars=available_cols, hue="algo_base", diag_kind="hist")
        plt.suptitle("ZX Graph Feature Space (Spider-Fused Level)", y=1.02)
        plt.savefig("feature_clustering.png", dpi=150)
        plt.show()
except ImportError:
    print("Install seaborn for clustering visualization: pip install seaborn")

## Observation Log

Fill this in as you explore — building intuition is the goal.

```
OBSERVATION LOG:

1. [algo] at [level]: noticed [pattern]
   Example: "grover_q3 at spider_fused: the oracle and diffusion
   operator produce symmetric Z-spider clusters connected by
   Hadamard edges — the two clusters are near-identical up to phase"

2. ...
```